In [ ]:
import autogen
autogen.__version__

In [ ]:
!pip install google

In [ ]:
! pip install google-genai

In [ ]:
! pip install vertexai

In [ ]:
! pip install gemini

In [ ]:
import os
from pathlib import Path

import agentops
from dotenv import load_dotenv
from autogen import ConversableAgent
from autogen.coding import CodeBlock, LocalCommandLineCodeExecutor


# ==========================================
# Load environment variables
# ==========================================

load_dotenv()

GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")
AGENTOPS_API_KEY = os.getenv("AGENTOPS_API_KEY")


# ==========================================
# Validate API keys
# ==========================================

if not GEMINI_API_KEY:
    raise ValueError("GEMINI_API_KEY is not configured")

if not AGENTOPS_API_KEY:
    raise ValueError("AGENTOPS_API_KEY is not configured")


# ==========================================
# Initialize AgentOps
# ==========================================

agentops.init(AGENTOPS_API_KEY)


# ==========================================
# Code execution directory
# ==========================================

work_dir = Path("geminichat")
work_dir.mkdir(exist_ok=True)


# ==========================================
# Create code executor
# ==========================================

executor = LocalCommandLineCodeExecutor(
    work_dir=work_dir
)


# ==========================================
# Agent 1 - Code Executor Agent
# ==========================================

code_executor_agent = ConversableAgent(
    name="code_executor_agent",

    llm_config=False,

    code_execution_config={
        "executor": executor,
    },

    human_input_mode="NEVER",
)


# ==========================================
# Agent 2 - Code Writer Agent
# ==========================================

code_writer_system_message = """
You have been given coding capability to solve tasks using Python code.
In the following cases, suggest python code (in a python coding block) or shell script (in a sh coding block) for the user 
to execute.
    1. When you need to collect info, use the code to output the info you need, for example, browse or search the web, 
    download/read a file, 
    print the content of a webpage or a file, get the current date/time, check the operating system. After sufficient 
    info is printed and the 
    task is ready to be solved based on your language skill, you can solve the task by yourself.
    2. When you need to perform some task with code, use the code to perform the task and output the result. Finish the
    task smartly.
Solve the task step by step if you need to. If a plan is not provided, explain your plan first. Be clear which step uses 
code, and which step uses your 
language skill.When using code, you must indicate the script type in the code block. The user cannot provide any other
feedback or perform any other 
action beyond executing the code you suggest. The user can't modify your code. So do not suggest incomplete code which
requires users to modify. 
Don't use a code block if it's not intended to be executed by the user.
If you want the user to save the code in a file before executing it, put # filename: <filename> inside the code block 
as the first line. Don't include multiple code blocks in one response. Do not ask users to copy and paste the result. 
Instead, use 'print' function for the output when relevant. Check the execution result returned by the user.
"""


code_writer_agent = ConversableAgent(
    name="code_writer",

    system_message=code_writer_system_message,

    llm_config={
        "config_list": [
            {
                "model": "Gemini 3.5 Flash-Lite",
                "api_key": GEMINI_API_KEY,
                "api_type": "google",
            }
        ]
    },

    code_execution_config=False,

    max_consecutive_auto_reply=2,

    human_input_mode="NEVER",
)

In [ ]:
import pprint

chat_result = code_executor_agent.initiate_chat(
    code_writer_agent, message="write python code for permutation and combination for the word UNIVERSE. use optimised way to calculate it .I want final result count"
)

pprint.pprint(chat_result)
agentops.end_session("Success")